# 1 Initialize the Database

All the code related to data management is in the `EnvironmentData` class. This makes life easier - for example: we can send the CatsUserID once and it becomes a class property. Then, when we call other operations we don't have to send this information again.

When you create a new instance of `EnvironmentData` and there is no database, it will pull historical data and initialize the database. 

In [1]:
# Clear prior data. 
import os, sys, shutil

# Add parent directory to Python path to import EnvironmentData.
sys.path.append(os.path.dirname(os.getcwd()))

# Get the EnvironmentData class.
from EnvironmentData import EnvironmentData 

# The project adds to existing data so we need to clear that data to get a solid test from scratch.
if os.path.exists('../data'):
    shutil.rmtree('../data')
    os.makedirs('../data')

# Initialize EnvironmentData. This will run the historical data pull.
envdt = EnvironmentData(
    #days_back = 365 * 2,
    days_back = 7,
    coris_enabled = True,
    licor_enabled = True,
    conserv_enabled = True, 
    testing = True,
    #testing = False,
    # Since we are running from the experiments/ folder, we need to tell the class to use the parent directory as home.
    home_directory = ".."
)

DEBUG: Enabled data sources: ['Conserv', 'Coris', 'LI-COR']


Gathering LI-COR readings: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████| 4/4 [00:03<00:00,  1.22it/s]


Detailed information is saved in the log:

In [2]:
# Detailed info is saved in the log.
with open('../data/EnvironmentData.log', 'r') as file:
    for line in file.read().splitlines()[:10]:
        print(line)

2025-12-04 08:33:01,637 - EnvironmentData - INFO - Initialized Conserv client with 5 customers
2025-12-04 08:33:01,638 - EnvironmentData - INFO - Enabled data sources: ['Conserv', 'Coris', 'LI-COR']
2025-12-04 08:33:01,638 - EnvironmentData - INFO - Fetching Conserv historical data for all customers
2025-12-04 08:33:01,638 - EnvironmentData - INFO - Fetching Conserv data for period: 1764257581 to 1764862381
2025-12-04 08:33:01,638 - EnvironmentData - INFO - Running in test mode - only processing first customer: 333
2025-12-04 08:33:01,681 - EnvironmentData - INFO - Fetching data for customer 333
2025-12-04 08:33:01,682 - EnvironmentData - INFO - Exporting chunk for customer 333: 2025-11-27 15:33:01+00:00 to 2025-12-04 15:33:01+00:00
2025-12-04 08:33:01,682 - EnvironmentData - INFO - Starting export for customer 333: 2025-11-27 15:33:01+00:00 to 2025-12-04 15:33:01+00:00
2025-12-04 08:33:01,682 - EnvironmentData - INFO - Conserv API POST https://api.conserv.io/v1/sensors/export headers=

This saves our intermediate data to `data/sensor_readings.parquet`. 

Initially, we leave the data mostly as-is. We'll clean, add formatted dates, consolidate readings from the same device, etc. when moving to analytical steps, this preserves the source data so we can always change our mind later about how we decide to view it. 

However, at this point we are taking care to standardize the data format between different API sources. 

There are just a few columns because this is only historical data. We'll bring in current data shortly, and that will add more columns. 

In [3]:
import polars
polars.read_parquet('../data/sensor_readings.parquet').filter(polars.col("Source") == "Coris").head()

SensorReadingUTC,QueryUTC,Source,DeviceID,DeviceName,SensorID,SensorName,SensorType,SensorReadingF,SensorReadingRh,Historical
i64,i32,str,str,str,str,str,str,f32,f32,bool
1764257581,1764862502,"""Coris""","""coris:12162""","""Peabody TH-L Mammal Hall D444""","""coris:21373""","""PYPM__0100104SET____ Temp YPM …","""Temperature""",68.720001,null,true
1764258481,1764862502,"""Coris""","""coris:12162""","""Peabody TH-L Mammal Hall D444""","""coris:21373""","""PYPM__0100104SET____ Temp YPM …","""Temperature""",68.68,null,true
1764259381,1764862502,"""Coris""","""coris:12162""","""Peabody TH-L Mammal Hall D444""","""coris:21373""","""PYPM__0100104SET____ Temp YPM …","""Temperature""",68.699997,null,true
1764260281,1764862502,"""Coris""","""coris:12162""","""Peabody TH-L Mammal Hall D444""","""coris:21373""","""PYPM__0100104SET____ Temp YPM …","""Temperature""",68.720001,null,true
1764261181,1764862502,"""Coris""","""coris:12162""","""Peabody TH-L Mammal Hall D444""","""coris:21373""","""PYPM__0100104SET____ Temp YPM …","""Temperature""",68.720001,null,true


In [4]:
polars.read_parquet('../data/sensor_readings.parquet').filter(polars.col("Source") == "LI-COR").head()

SensorReadingUTC,QueryUTC,Source,DeviceID,DeviceName,SensorID,SensorName,SensorType,SensorReadingF,SensorReadingRh,Historical
i64,i32,str,str,str,str,str,str,f32,f32,bool
1764258300,1764862519,"""LI-COR""","""licor:22202142""","""RX Station 1""","""licor:22202142-22179175-1""","""RX Station 1_Temperature""","""Temperature""",70.45092,null,true
1764259200,1764862519,"""LI-COR""","""licor:22202142""","""RX Station 1""","""licor:22202142-22179175-1""","""RX Station 1_Temperature""","""Temperature""",70.45092,null,true
1764260100,1764862519,"""LI-COR""","""licor:22202142""","""RX Station 1""","""licor:22202142-22179175-1""","""RX Station 1_Temperature""","""Temperature""",70.373695,null,true
1764261000,1764862519,"""LI-COR""","""licor:22202142""","""RX Station 1""","""licor:22202142-22179175-1""","""RX Station 1_Temperature""","""Temperature""",70.335091,null,true
1764261900,1764862519,"""LI-COR""","""licor:22202142""","""RX Station 1""","""licor:22202142-22179175-1""","""RX Station 1_Temperature""","""Temperature""",70.373695,null,true


In [5]:
polars.read_parquet('../data/sensor_readings.parquet').filter(polars.col("Source") == "Conserv").head()

SensorReadingUTC,QueryUTC,Source,DeviceID,DeviceName,SensorID,SensorName,SensorType,SensorReadingF,SensorReadingRh,Historical
i64,i32,str,str,str,str,str,str,f32,f32,bool
1764258437,1764862382,"""Conserv""","""conserv:333:c009096""","""BFAST__BS___________""","""conserv:333:c009096:Temperatur…","""conserv:333:c009096:Temperatur…","""Temperature""",69.115997,null,true
1764259337,1764862382,"""Conserv""","""conserv:333:c009096""","""BFAST__BS___________""","""conserv:333:c009096:Temperatur…","""conserv:333:c009096:Temperatur…","""Temperature""",69.188004,null,true
1764260237,1764862382,"""Conserv""","""conserv:333:c009096""","""BFAST__BS___________""","""conserv:333:c009096:Temperatur…","""conserv:333:c009096:Temperatur…","""Temperature""",69.475998,null,true
1764261137,1764862382,"""Conserv""","""conserv:333:c009096""","""BFAST__BS___________""","""conserv:333:c009096:Temperatur…","""conserv:333:c009096:Temperatur…","""Temperature""",69.944,null,true
1764262037,1764862382,"""Conserv""","""conserv:333:c009096""","""BFAST__BS___________""","""conserv:333:c009096:Temperatur…","""conserv:333:c009096:Temperatur…","""Temperature""",70.232002,null,true


# 2 Get Current Readings

Now we can start gathering and appending readings. There is a function `get_current_readings` that is run throughout the day, every 10 minutes for example. This function creates a parquet file at `data/new-readings` with the UTC as a filename. At the end of the day, all these readings will be consolidated into the database. 

Here is a sample of the readings:

In [6]:
# Wait 15 minutes to allow a new Conserv reading.
import time
time.sleep(15 * 60)  # Wait 15 minutes (900 seconds)

envdt.get_current_readings()

# Data is read into new-readings folder for consolidation at the end of the day.
import os
filename = os.listdir('../data/new-readings')[0]
print(filename)
polars.read_parquet('../data/new-readings/' + filename).sample(5)

Gathering Conserv current readings: 100%|███████████████████████████████████████████████████████████████████████████████████████████████| 1/1 [01:46<00:00, 106.25s/it]


✓ Conserv: 60 records
✓ Coris: 6 records
✓ LI-COR: 8 records
1764863422.parquet


SensorReadingUTC,QueryUTC,Source,DeviceID,DeviceName,SensorID,SensorName,SensorType,SensorReadingF,SensorReadingRh,Historical
i64,i32,str,str,str,str,str,str,f32,f32,bool
1764863416,1764863423,"""Conserv""","""conserv:333:c009023""","""BYCBA_0200212_NE____""","""conserv:333:c009023:Temperatur…","""conserv:333:c009023:Temperatur…","""Temperature""",70.232002,null,false
1764863531,1764863530,"""LI-COR""","""licor:22202142""","""RX Station 1""","""licor:22202142-22179175-2""","""RX Station 1_RH""","""RH""",null,42.484169,false
1764862855,1764863423,"""Conserv""","""conserv:333:c008903""","""BYCBA_030030107_S___""","""conserv:333:c008903:RH""","""conserv:333:c008903:RH""","""RH""",null,45.82,false
1764863187,1764863423,"""Conserv""","""conserv:333:c008734""","""BYCBA_040040113_S___""","""conserv:333:c008734:RH""","""conserv:333:c008734:RH""","""RH""",null,44.220001,false
1764863284,1764863423,"""Conserv""","""conserv:333:c008733""","""BYCBA_040040108_E___""","""conserv:333:c008733:RH""","""conserv:333:c008733:RH""","""RH""",null,43.759998,false


# 3 Consolidate Readings

At the end of the day, new readings will be consolidated into the table. At the same time, the analytical tables will be generated. 

Analytical tables include:

* `device_readings.parquet`: Sensor readings reorganized to one row per Device and UTC, with measurements across columns vs measurements across rows. 
* `sensors.parquet`: Information about the unique sensors. Includes information extracted from SensorName. Join this to Sensors during analysis to enhance with Building, Room, Direction, etc.
* `devices.parquet`: Information about unique devices. Includes information extracted from SensorName. 
* `utcs.parquet`: Information related to the UTC times in various datasets. Join to Sensors or Devices to enhance with Date, Time, Year, Hour, Weekday, etc.
* `sensor_readings_daily.parquet`: Example of sensor readings summarized to the daily level which reduces row count by 99.3% for even faster queries.
* `device_readings_daily.parquet`: Example of device readings summarized to the daily level which reduces row count by 99.3% for even faster queries. 

We fully re-generate analytical tables during each consolidation. The data is small enough that this is a fairly quick process, so re-running it in full each time will make it easy to ensure consistency as we expand and change the project. 

In [7]:
# To consolidate these into the database, run consolidate_readings.
envdt.consolidate_readings()

# New-readings files are gone now.
# They get deleted each day to confirm that they have been loaded into the database and prepare for the next consolidation.
if os.path.exists('../data/new-readings'):
    print(os.listdir('../data/new-readings'))

[]


In [8]:
# Use this to re-run if you change the consolidation code.

# from EnvironmentData import EnvironmentData 
# envdt = EnvironmentData(
#     #days_back = 365 * 2,
#     days_back = 7,
#     coris_enabled = True,
#     licor_enabled = True,
#     conserv_enabled = True, 
#     #testing = True,
#     testing = False,
#     # Since we are running from the experiments/ folder, we need to tell the class to use the parent directory as home.
#     home_directory = ".."
# )
# envdt.close()
# del envdt

**^^ We want this to be empty** since we have consolidated new readings into the historical data. 

Once we are done working with data intake/processing, we close the class to release the file lock on the log file.

In [9]:
# When done, close the connection to the logs. 
envdt.close()

Let's look at the data we have now:

In [10]:
# Sensor Readings
# The first rows will be missing the extra fields like HexGatewayMac, etc.
#   I am pulling in some extra fields like DeviceID and DeviceName so we have that by historical. 
#   But some don't make sense to  backfill so they'll be null.
sensor_readings = polars.read_parquet('../data/sensor_readings.parquet')
sensor_readings.head()

SensorReadingUTC,QueryUTC,Source,DeviceID,DeviceName,SensorID,SensorName,SensorType,SensorReadingF,SensorReadingRh,SensorReadingUTC_SecondsFromPrior,Historical
i64,i32,str,str,str,str,str,str,f32,f32,i64,bool
1764258135,1764862382,"""Conserv""","""conserv:333:c008706""","""BYCBA_0400410__N____""","""conserv:333:c008706:RH""","""conserv:333:c008706:RH""","""RH""",null,50.869999,null,true
1764259035,1764862382,"""Conserv""","""conserv:333:c008706""","""BYCBA_0400410__N____""","""conserv:333:c008706:RH""","""conserv:333:c008706:RH""","""RH""",null,49.540001,null,true
1764259935,1764862382,"""Conserv""","""conserv:333:c008706""","""BYCBA_0400410__N____""","""conserv:333:c008706:RH""","""conserv:333:c008706:RH""","""RH""",null,48.720001,null,true
1764260835,1764862382,"""Conserv""","""conserv:333:c008706""","""BYCBA_0400410__N____""","""conserv:333:c008706:RH""","""conserv:333:c008706:RH""","""RH""",null,49.470001,null,true
1764261735,1764862382,"""Conserv""","""conserv:333:c008706""","""BYCBA_0400410__N____""","""conserv:333:c008706:RH""","""conserv:333:c008706:RH""","""RH""",null,48.970001,null,true


In [11]:
# Recent rows will have the full data, aside from nulls due to a sensor not providing a reading type.
sensor_readings.tail()

SensorReadingUTC,QueryUTC,Source,DeviceID,DeviceName,SensorID,SensorName,SensorType,SensorReadingF,SensorReadingRh,SensorReadingUTC_SecondsFromPrior,Historical
i64,i32,str,str,str,str,str,str,f32,f32,i64,bool
1764855000,1764862519,"""LI-COR""","""licor:22202142""","""RX Station 1""","""licor:22202142-22179175-1""","""RX Station 1_Temperature""","""Temperature""",70.56675,null,null,true
1764855900,1764862519,"""LI-COR""","""licor:22202142""","""RX Station 1""","""licor:22202142-22179175-1""","""RX Station 1_Temperature""","""Temperature""",70.721191,null,null,true
1764856800,1764862519,"""LI-COR""","""licor:22202142""","""RX Station 1""","""licor:22202142-22179175-1""","""RX Station 1_Temperature""","""Temperature""",70.643967,null,null,true
1764857700,1764862519,"""LI-COR""","""licor:22202142""","""RX Station 1""","""licor:22202142-22179175-1""","""RX Station 1_Temperature""","""Temperature""",70.875633,null,null,true
1764863531,1764863530,"""LI-COR""","""licor:22202142""","""RX Station 1""","""licor:22202142-22179175-1""","""RX Station 1_Temperature""","""Temperature""",70.875633,null,null,false


In [12]:
# Device Readings.
device_readings = polars.read_parquet('../data/device_readings.parquet')
device_readings.head()

Source,DeviceID,DeviceName,Sensors,SensorNames,SensorTypes,SensorReadingUTC,QueryUTC,Historical,SensorReadingF,SensorReadingRh
str,str,str,str,str,str,i64,i32,bool,f32,f32
"""Conserv""","""conserv:333:c008706""","""BYCBA_0400410__N____""","""conserv:333:c008706:RH|conserv…","""conserv:333:c008706:RH|conserv…","""RH|Temperature""",1764258135,1764862382,true,69.673996,50.869999
"""Conserv""","""conserv:333:c008706""","""BYCBA_0400410__N____""","""conserv:333:c008706:RH|conserv…","""conserv:333:c008706:RH|conserv…","""RH|Temperature""",1764259035,1764862382,true,69.584,49.540001
"""Conserv""","""conserv:333:c008706""","""BYCBA_0400410__N____""","""conserv:333:c008706:RH|conserv…","""conserv:333:c008706:RH|conserv…","""RH|Temperature""",1764259935,1764862382,true,69.619995,48.720001
"""Conserv""","""conserv:333:c008706""","""BYCBA_0400410__N____""","""conserv:333:c008706:RH|conserv…","""conserv:333:c008706:RH|conserv…","""RH|Temperature""",1764260835,1764862382,true,69.710007,49.470001
"""Conserv""","""conserv:333:c008706""","""BYCBA_0400410__N____""","""conserv:333:c008706:RH|conserv…","""conserv:333:c008706:RH|conserv…","""RH|Temperature""",1764261735,1764862382,true,69.962006,48.970001


In [13]:
# Sensors
sensors = polars.read_parquet('../data/sensors.parquet')
sensors.head()

Source,SensorID,SensorName,SensorType,DeviceID
str,str,str,str,str
"""Conserv""","""conserv:333:c008706:RH""","""conserv:333:c008706:RH""","""RH""","""conserv:333:c008706"""
"""Conserv""","""conserv:333:c008706:Temperatur…","""conserv:333:c008706:Temperatur…","""Temperature""","""conserv:333:c008706"""
"""Conserv""","""conserv:333:c008733:RH""","""conserv:333:c008733:RH""","""RH""","""conserv:333:c008733"""
"""Conserv""","""conserv:333:c008733:Temperatur…","""conserv:333:c008733:Temperatur…","""Temperature""","""conserv:333:c008733"""
"""Conserv""","""conserv:333:c008734:RH""","""conserv:333:c008734:RH""","""RH""","""conserv:333:c008734"""


In [14]:
# Devices. 
devices = polars.read_parquet('../data/devices.parquet')
devices.head()

Source,DeviceID,DeviceName,SensorIDs,SensorNames,SensorTypes
str,str,str,str,str,str
"""Conserv""","""conserv:333:c009072""","""BBARCH0100001_______""","""conserv:333:c009072:RH|conserv…","""conserv:333:c009072:RH|conserv…","""RH|Temperature"""
"""Conserv""","""conserv:333:c009073""","""BBARCHB100116_______""","""conserv:333:c009073:RH|conserv…","""conserv:333:c009073:RH|conserv…","""RH|Temperature"""
"""Conserv""","""conserv:333:c009081""","""BCSC__01H103________""","""conserv:333:c009081:RH|conserv…","""conserv:333:c009081:RH|conserv…","""RH|Temperature"""
"""Conserv""","""conserv:333:c009096""","""BFAST__BS___________""","""conserv:333:c009096:RH|conserv…","""conserv:333:c009096:RH|conserv…","""RH|Temperature"""
"""Conserv""","""conserv:333:c009025""","""BYCBA_0100101_N_____""","""conserv:333:c009025:RH|conserv…","""conserv:333:c009025:RH|conserv…","""RH|Temperature"""


In [15]:
# UTC Date/Time Info
utcs = polars.read_parquet('../data/utcs.parquet').head()
utcs.head()

UTC,datetime_utc,datetime_est,date,time,year,month,day_of_week,day_of_week_monday1_sunday7,hour_24,hour_12,am_pm
i64,datetime[μs],"datetime[μs, America/New_York]",date,time,i32,i8,str,i8,i8,i8,str
1764360204,2025-11-28 13:03:24,2025-11-28 08:03:24 EST,2025-11-28,08:03:24,2025,11,"""Friday""",5,8,8,"""AM"""
1764622350,2025-12-01 13:52:30,2025-12-01 08:52:30 EST,2025-12-01,08:52:30,2025,12,"""Monday""",1,8,8,"""AM"""
1764753439,2025-12-03 02:17:19,2025-12-02 21:17:19 EST,2025-12-02,21:17:19,2025,12,"""Tuesday""",2,21,9,"""PM"""
1764753463,2025-12-03 02:17:43,2025-12-02 21:17:43 EST,2025-12-02,21:17:43,2025,12,"""Tuesday""",2,21,9,"""PM"""
1764753476,2025-12-03 02:17:56,2025-12-02 21:17:56 EST,2025-12-02,21:17:56,2025,12,"""Tuesday""",2,21,9,"""PM"""


In [16]:
# Daily Sensor Readings
# Averages are calculated by summing the "sum" and "row_count" columns and dividing to get the average. 
sensor_readings_daily = polars.read_parquet('../data/sensor_readings_daily.parquet')
sensor_readings_daily.head()

Source,date,SensorID,row_count,SensorReadingF_sum,SensorReadingRh_sum,SensorReadingF_min,SensorReadingRh_min,SensorReadingF_max,SensorReadingRh_max
str,date,str,u32,f32,f32,f32,f32,f32,f32
"""Conserv""",2025-12-04,"""conserv:333:c008733:RH""",1,0.0,43.759998,null,43.759998,null,43.759998
"""Conserv""",2025-12-04,"""conserv:333:c008733:Temperatur…",1,69.854004,0.0,69.854004,null,69.854004,null
"""Conserv""",2025-12-04,"""conserv:333:c008734:RH""",2,0.0,89.199997,null,44.220001,null,44.98
"""Conserv""",2025-12-04,"""conserv:333:c008734:Temperatur…",2,138.808014,0.0,69.386002,null,69.422005,null
"""Conserv""",2025-12-04,"""conserv:333:c008784:RH""",1,0.0,44.549999,null,44.549999,null,44.549999


In [17]:
# Daily Device Readings
# Averages are calculated by summing the "sum" and "row_count" columns and dividing to get the average. 
device_readings_daily = polars.read_parquet('../data/device_readings_daily.parquet')
device_readings_daily.head()

Source,date,DeviceID,row_count,SensorReadingF_sum,SensorReadingRh_sum,SensorReadingF_min,SensorReadingRh_min,SensorReadingF_max,SensorReadingRh_max
str,date,str,u32,f32,f32,f32,f32,f32,f32
"""Conserv""",2025-12-04,"""conserv:333:c008733""",1,69.854004,43.759998,69.854004,43.759998,69.854004,43.759998
"""Conserv""",2025-12-04,"""conserv:333:c008734""",2,138.808014,89.199997,69.386002,44.220001,69.422005,44.98
"""Conserv""",2025-12-04,"""conserv:333:c008784""",1,69.566002,44.549999,69.566002,44.549999,69.566002,44.549999
"""Conserv""",2025-12-04,"""conserv:333:c008785""",2,138.574005,90.539993,69.223999,44.919998,69.350006,45.619999
"""Conserv""",2025-12-04,"""conserv:333:c008786""",2,140.373993,95.029999,70.07,47.299999,70.304001,47.73


In [18]:
# Differentiate historical vs. cron readings by filtering on Historical = true.
import duckdb
duckdb.sql("""
    SELECT *
    FROM read_parquet('../data/device_readings.parquet') 
    WHERE Historical
    LIMIT 5
""").to_df()

,Source,DeviceID,DeviceName,Sensors,SensorNames,SensorTypes,SensorReadingUTC,QueryUTC,Historical,SensorReadingF,SensorReadingRh
0,Conserv,conserv:333:c008706,BYCBA_0400410__N____,conserv:333:c008706:RH|conserv:333:c008706:Tem...,conserv:333:c008706:RH|conserv:333:c008706:Tem...,RH|Temperature,1764258135,1764862382,True,69.673996,50.869999
1,Conserv,conserv:333:c008706,BYCBA_0400410__N____,conserv:333:c008706:RH|conserv:333:c008706:Tem...,conserv:333:c008706:RH|conserv:333:c008706:Tem...,RH|Temperature,1764259035,1764862382,True,69.584000,49.540001
2,Conserv,conserv:333:c008706,BYCBA_0400410__N____,conserv:333:c008706:RH|conserv:333:c008706:Tem...,conserv:333:c008706:RH|conserv:333:c008706:Tem...,RH|Temperature,1764259935,1764862382,True,69.619995,48.720001
3,Conserv,conserv:333:c008706,BYCBA_0400410__N____,conserv:333:c008706:RH|conserv:333:c008706:Tem...,conserv:333:c008706:RH|conserv:333:c008706:Tem...,RH|Temperature,1764260835,1764862382,True,69.710007,49.470001
4,Conserv,conserv:333:c008706,BYCBA_0400410__N____,conserv:333:c008706:RH|conserv:333:c008706:Tem...,conserv:333:c008706:RH|conserv:333:c008706:Tem...,RH|Temperature,1764261735,1764862382,True,69.962006,48.970001


In [19]:
duckdb.sql("""SELECT DISTINCT
    sr.Source,
    u.datetime_est
FROM '../data/sensor_readings.parquet' sr
LEFT JOIN '../data/utcs.parquet' u 
    ON sr.SensorReadingUTC = u.utc
WHERE sr.Source = 'LI-COR'
ORDER BY sr.SensorReadingUTC
""").to_df()

,Source,datetime_est
0,LI-COR,2025-11-27 01:45:00-07:00
1,LI-COR,2025-11-27 02:00:00-07:00
2,LI-COR,2025-11-27 02:15:00-07:00
3,LI-COR,2025-11-27 02:30:00-07:00
4,LI-COR,2025-11-27 02:45:00-07:00
...,...,...
663,LI-COR,2025-12-03 23:30:00-07:00
664,LI-COR,2025-12-03 23:45:00-07:00
665,LI-COR,2025-12-04 00:00:00-07:00
666,LI-COR,2025-12-04 00:15:00-07:00


# Validation

There are two diagnostic files we can review to see if there are alerts or errors. 

In [20]:
# Read ../data/validation-results.csv
import pandas as pd
validation_results = pd.read_csv('../data/validation-results.csv')
validation_results

,run_datetime_est,run_utc,test_name,result,details
0,2025-12-04 10:52:11 EST,1764863531,required_columns_present,PASS,All 11 required columns are present in sensor ...
1,2025-12-04 10:52:11 EST,1764863531,column_data_types,PASS,All columns have the expected data types (e.g....
2,2025-12-04 10:52:11 EST,1764863531,non_null_values,PASS,Every sensor reading row has at least one non-...
3,2025-12-04 10:52:11 EST,1764863531,no_duplicate_readings,PASS,No duplicate readings found. Each sensor has u...
4,2025-12-04 10:52:11 EST,1764863531,sensor_name_consistency,PASS,All sensors have consistent names across all t...
5,2025-12-04 10:52:11 EST,1764863531,reading_interval_check,PASS,All consecutive readings are within 15 minutes...
6,2025-12-04 10:52:11 EST,1764863531,data_gaps_Conserv,WARN,Found 160 gaps in Conserv data where readings ...
7,2025-12-04 10:52:11 EST,1764863531,data_gaps_Coris,WARN,Found 6 gaps in Coris data where readings are ...
8,2025-12-04 10:52:11 EST,1764863531,data_gaps_LI-COR,WARN,Found 2 gaps in LI-COR data where readings are...
9,2025-12-04 10:52:11 EST,1764863531,alerts_Coris,PASS,No alerts triggered for Coris. All sensor read...


In [21]:
# Validation results that did not pass.
validation_results[validation_results['result'] != "PASS"]

,run_datetime_est,run_utc,test_name,result,details
6,2025-12-04 10:52:11 EST,1764863531,data_gaps_Conserv,WARN,Found 160 gaps in Conserv data where readings ...
7,2025-12-04 10:52:11 EST,1764863531,data_gaps_Coris,WARN,Found 6 gaps in Coris data where readings are ...
8,2025-12-04 10:52:11 EST,1764863531,data_gaps_LI-COR,WARN,Found 2 gaps in LI-COR data where readings are...


In [22]:
# Read ../data/validation-detail.parquet (contains DATA_GAP events)
detail = pd.read_parquet('../data/validation-detail.parquet')
detail.sample(10).sort_values(by='event_utc')

,event,Source,SensorID,SensorName,event_utc,event_datetime_est,event_end_utc,event_end_datetime_est,gap_minutes,detected_utc,detected_datetime_est
133,DATA_GAP,Conserv,conserv:333:c009072:Temperature,conserv:333:c009072:Temperature,1764297376,2025-11-27 21:36:16 EST,1764299176,2025-11-27 22:06:16 EST,30.0,1764863531,2025-12-04 10:52:11 EST
104,DATA_GAP,Conserv,conserv:333:c009063:RH,conserv:333:c009063:RH,1764330675,2025-11-28 06:51:15 EST,1764332475,2025-11-28 07:21:15 EST,30.0,1764863531,2025-12-04 10:52:11 EST
82,DATA_GAP,Conserv,conserv:333:c009023:Temperature,conserv:333:c009023:Temperature,1764501617,2025-11-30 06:20:17 EST,1764503417,2025-11-30 06:50:17 EST,30.0,1764863531,2025-12-04 10:52:11 EST
94,DATA_GAP,Conserv,conserv:333:c009057:RH,conserv:333:c009057:RH,1764704510,2025-12-02 14:41:50 EST,1764706310,2025-12-02 15:11:50 EST,30.0,1764863531,2025-12-04 10:52:11 EST
34,DATA_GAP,Conserv,conserv:333:c008784:Temperature,conserv:333:c008784:Temperature,1764724766,2025-12-02 20:19:26 EST,1764726566,2025-12-02 20:49:26 EST,30.0,1764863531,2025-12-04 10:52:11 EST
30,DATA_GAP,Conserv,conserv:333:c008784:RH,conserv:333:c008784:RH,1764724766,2025-12-02 20:19:26 EST,1764726566,2025-12-02 20:49:26 EST,30.0,1764863531,2025-12-04 10:52:11 EST
102,DATA_GAP,Conserv,conserv:333:c009057:Temperature,conserv:333:c009057:Temperature,1764764810,2025-12-03 07:26:50 EST,1764766609,2025-12-03 07:56:49 EST,30.0,1764863531,2025-12-04 10:52:11 EST
124,DATA_GAP,Conserv,conserv:333:c009069:Temperature,conserv:333:c009069:Temperature,1764771840,2025-12-03 09:24:00 EST,1764773640,2025-12-03 09:54:00 EST,30.0,1764863531,2025-12-04 10:52:11 EST
44,DATA_GAP,Conserv,conserv:333:c008786:Temperature,conserv:333:c008786:Temperature,1764840630,2025-12-04 04:30:30 EST,1764842430,2025-12-04 05:00:30 EST,30.0,1764863531,2025-12-04 10:52:11 EST
67,DATA_GAP,Conserv,conserv:333:c008924:Temperature,conserv:333:c008924:Temperature,1764861569,2025-12-04 10:19:29 EST,1764863369,2025-12-04 10:49:29 EST,30.0,1764863531,2025-12-04 10:52:11 EST


In [23]:
# Group data gaps by Source.
detail.groupby(['Source', 'event']).size().reset_index(name='count')

,Source,event,count
0,Conserv,DATA_GAP,160
1,Coris,DATA_GAP,6
2,LI-COR,DATA_GAP,2


Now you are ready to move onto analysis to get human-readable results (not indexed by UTC timestamps). See 2-examples-analysis.ipynb.